Main imports and setup

In [ ]:
import os
import json
from typing import List, Dict, Any, Optional
import pandas as pd
import numpy as np
from pathlib import Path

# langChain
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.schema import Document
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough

# LangGraph
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

# Qdrant
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

# ragas
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
    answer_correctness
)
from datasets import Dataset

# Cohere for semantic similarity
import cohere

# Set up environment variables
# load the .env file
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["COHERE_API_KEY"] = os.getenv("COHERE_API_KEY")

# Initialize models
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings()
co = cohere.Client(os.environ["COHERE_API_KEY"])

Data loading and processing

In [ ]:
def load_pdf_documents(data_dir: str = "data") -> List[Document]:
    documents = []
    data_path = Path(data_dir)
    
    for pdf_file in data_path.glob("*.pdf"):
        loader = PyMuPDFLoader(str(pdf_file))
        documents.extend(loader.load())
    
    return documents

# show how many documents and words we have
documents = load_pdf_documents()
print(f"Loaded {len(documents)} documents")
print(f"Total words: {sum(len(doc.page_content.split()) for doc in documents)}")


Loaded 269 documents
Total words: 136001


chunking

In [49]:
def create_naive_chunks(documents: List[Document], chunk_size: int = 1000, chunk_overlap: int = 200) -> List[Document]:
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    chunks = []
    for doc in documents:
        doc_chunks = text_splitter.split_documents([doc])
        chunks.extend(doc_chunks)
    
    return chunks

naive_chunks = create_naive_chunks(documents)
print(f"Created {len(naive_chunks)} naive chunks")

Created 1102 naive chunks


semantic chunking strategy

In [51]:
def split_into_sentences(text: str) -> List[str]:
    import re
    sentences = re.split(r'[.!?]+', text)
    return [s.strip() for s in sentences if s.strip()]

def create_semantic_chunks(
    documents: List[Document], 
    similarity_threshold: float = 0.7, 
    max_chunk_size: int = 1000
) -> List[Document]:
    semantic_chunks = []

    for doc in documents:
        text = doc.page_content
        sentences = split_into_sentences(text)
        if not sentences:
            continue

        # Batch embed all sentences in this document
        try:
            sentence_embeddings = embeddings.embed_documents(sentences)
        except Exception as e:
            print(f"Embedding error: {e}")
            continue

        current_chunk = [sentences[0]]
        current_size = len(sentences[0].split())
        last_idx = 0

        for idx in range(1, len(sentences)):
            sentence = sentences[idx]
            sentence_size = len(sentence.split())

            # Check size limit first
            if current_size + sentence_size > max_chunk_size and current_chunk:
                chunk_text = " ".join(current_chunk)
                semantic_chunks.append(Document(
                    page_content=chunk_text,
                    metadata=doc.metadata
                ))
                current_chunk = [sentence]
                current_size = sentence_size
                last_idx = idx
                continue

            # Compute cosine similarity using precomputed embeddings
            last_embedding = sentence_embeddings[last_idx]
            curr_embedding = sentence_embeddings[idx]
            similarity = np.dot(last_embedding, curr_embedding) / (
                np.linalg.norm(last_embedding) * np.linalg.norm(curr_embedding)
            )

            if similarity >= similarity_threshold:
                current_chunk.append(sentence)
                current_size += sentence_size
            else:
                chunk_text = " ".join(current_chunk)
                semantic_chunks.append(Document(
                    page_content=chunk_text,
                    metadata=doc.metadata
                ))
                current_chunk = [sentence]
                current_size = sentence_size
                last_idx = idx

        # Add final chunk
        if current_chunk:
            chunk_text = " ".join(current_chunk)
            semantic_chunks.append(Document(
                page_content=chunk_text,
                metadata=doc.metadata
            ))

    return semantic_chunks

semantic_chunks = create_semantic_chunks(documents)
print(f"Created {len(semantic_chunks)} semantic chunks")

Created 292 semantic chunks


In [76]:
difference = len(naive_chunks) - len(semantic_chunks)

print(f"Difference: {difference} chunks")
print(f"Percentage of reduction: {float(difference / len(naive_chunks) * 100):.2f}%")

Difference: 810 chunks
Percentage of reduction: 73.50%


vector storage

In [52]:
def create_vector_store(chunks: List[Document], collection_name: str) -> Qdrant:
    # Use persistent storage
    client = QdrantClient(path=f"./qdrant_data_{collection_name}")
    
    # Get embedding dimension dynamically
    test_embedding = embeddings.embed_query("test")
    embedding_dim = len(test_embedding)
    
    try:
        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=embedding_dim, distance=Distance.COSINE),
        )
    except Exception:
        # Collection might already exist
        pass
    
    vector_store = QdrantVectorStore(
        client=client,
        collection_name=collection_name,
        embedding=embeddings,
    )
    
    # Actually add documents to the vector store
    vector_store.add_documents(chunks)
    
    return vector_store

naive_vector_store = create_vector_store(naive_chunks, "naive_chunks")
semantic_vector_store = create_vector_store(semantic_chunks, "semantic_chunks")

print("Vector stores created successfully")

Vector stores created successfully


rag chain

In [53]:
def create_rag_chain(vector_store: Qdrant) -> Any:
    
    retriever = vector_store.as_retriever(search_kwargs={"k": 5})
    
    template = """You are a helpful assistant that answers questions based on the provided context.
    
    Context: {context}
    
    Question: {question}
    
    Answer the question based on the context provided. If the context doesn't contain enough information to answer the question, say so."""
    
    prompt = ChatPromptTemplate.from_template(template)
    
    chain = (
        {"context": retriever, "question": RunnablePassthrough()}
        | prompt
        | llm
    )
    
    return chain

naive_rag_chain = create_rag_chain(naive_vector_store)
semantic_rag_chain = create_rag_chain(semantic_vector_store)

print("RAG chains created successfully")

RAG chains created successfully


langgraph + rag

In [ ]:
def create_langgraph_rag(vector_store: Qdrant) -> StateGraph:
    
    from typing import TypedDict, Annotated
    from langgraph.graph.message import add_messages
    
    class GraphState(TypedDict):
        messages: Annotated[list, add_messages]
        context: list
        question: str
    
    def retrieve_context(state: GraphState) -> GraphState:
        question = state["question"]
        retriever = vector_store.as_retriever(search_kwargs={"k": 5})
        docs = retriever.invoke(question) 
        return {"context": [doc.page_content for doc in docs]}
    
    def generate_answer(state: GraphState) -> GraphState:
        context = "\n\n".join(state["context"])
        question = state["question"]
        
        template = """You are a helpful assistant that answers questions based on the provided context.
        
        Context: {context}
        
        Question: {question}
        
        Answer the question based on the context provided. If the context doesn't contain enough information to answer the question, say so."""
        
        prompt = ChatPromptTemplate.from_template(template)
        chain = prompt | llm
        
        response = chain.invoke({"context": context, "question": question})
        
        return {"messages": [response]}
    
    workflow = StateGraph(GraphState)
    
    workflow.add_node("retrieve", retrieve_context)
    workflow.add_node("generate", generate_answer)
    
    workflow.set_entry_point("retrieve")
    workflow.add_edge("retrieve", "generate")
    workflow.add_edge("generate", END)
    
    app = workflow.compile()
    
    return app

naive_langgraph_rag = create_langgraph_rag(naive_vector_store)
semantic_langgraph_rag = create_langgraph_rag(semantic_vector_store)

print("LangGraph RAG applications created successfully")

LangGraph RAG applications created successfully


synthetic dataset generation

In [80]:
def generate_synthetic_dataset() -> Dataset:

    questions = [
        "What is the Federal Pell Grant Program?",
        "How do I apply for a Direct Loan?",
        "What is the maximum Pell Grant amount?",
        "What are the eligibility requirements for federal student aid?",
        "How is the cost of attendance calculated?",
        "What is the difference between subsidized and unsubsidized loans?",
        "How do I verify my financial aid application?",
        "What is the academic calendar for financial aid?",
        "How do I package financial aid awards?",
        "What documents do I need for verification?"
    ]
    
    synthetic_data = []
    
    for question in questions:
        naive_response = naive_rag_chain.invoke(question)
        naive_answer = naive_response.content
        
        semantic_response = semantic_rag_chain.invoke(question)
        semantic_answer = semantic_response.content
        
        naive_retriever = naive_vector_store.as_retriever(search_kwargs={"k": 5})
        semantic_retriever = semantic_vector_store.as_retriever(search_kwargs={"k": 5})
        
        naive_context = naive_retriever.invoke(question) 
        semantic_context = semantic_retriever.invoke(question)
        
        synthetic_data.append({
            "question": question,
            "naive_answer": naive_answer,
            "semantic_answer": semantic_answer,
            "naive_context": [doc.page_content for doc in naive_context],
            "semantic_context": [doc.page_content for doc in semantic_context]
        })
    
    return Dataset.from_list(synthetic_data)

synthetic_dataset = generate_synthetic_dataset()
print(f"Generated synthetic dataset with {len(synthetic_dataset)} examples")

# print pandas table of the synthetic dataset
df = pd.DataFrame(synthetic_dataset)
df.head()

Generated synthetic dataset with 10 examples


,question,naive_answer,semantic_answer,naive_context,semantic_context
0,What is the Federal Pell Grant Program?,The Federal Pell Grant Program is a financial ...,The Federal Pell Grant Program is a financial ...,[Volume 7\nThe Federal Pell Grant Program\nInt...,[Volume 7\nThe Federal Pell Grant Program\nInt...
1,How do I apply for a Direct Loan?,"To apply for a Direct Loan, you should complet...",The provided context does not contain specific...,[complete the Direct PLUS Loan Application (fo...,[Chapter 1\nStudent and Parent Eligibility for...
2,What is the maximum Pell Grant amount?,The maximum Pell Grant amount mentioned in the...,The maximum Pell Grant amount is determined by...,"[programs will be $7,500:\n$15,000\n- 0\n- $7,...","[Award, for a maximum total of $10,500 Adding ..."
3,What are the eligibility requirements for fede...,The eligibility requirements for federal stude...,The eligibility requirements for federal stude...,[Chapter 1\nStudent and Parent Eligibility for...,[Examples of eligible noncitizen categories ar...
4,How is the cost of attendance calculated?,The cost of attendance (COA) is calculated bas...,The cost of attendance (COA) is calculated bas...,[Chapter 2\nCost of Attendance (Budget)\nAward...,[Chapter 2\nCost of Attendance (Budget)\nAward...


ragas evaluation

In [81]:
def prepare_ragas_dataset(synthetic_data: Dataset, strategy: str) -> Dataset:
    
    if strategy == "naive":
        answers = synthetic_data["naive_answer"]
        contexts = synthetic_data["naive_context"]
    else:
        answers = synthetic_data["semantic_answer"]
        contexts = synthetic_data["semantic_context"]
    
    ragas_data = []
    for i in range(len(synthetic_data)):
        # Add reference column for context_precision metric
        ragas_data.append({
            "question": synthetic_data["question"][i],
            "answer": answers[i],
            "contexts": contexts[i],
            "reference": answers[i]  # Use answer as reference for context_precision
        })
    
    return Dataset.from_list(ragas_data)

naive_ragas_dataset = prepare_ragas_dataset(synthetic_dataset, "naive")
semantic_ragas_dataset = prepare_ragas_dataset(synthetic_dataset, "semantic")

print("Ragas datasets prepared successfully")

# pandas table
df = pd.DataFrame(naive_ragas_dataset)
df.head()

Ragas datasets prepared successfully


,question,answer,contexts,reference
0,What is the Federal Pell Grant Program?,The Federal Pell Grant Program is a financial ...,[Volume 7\nThe Federal Pell Grant Program\nInt...,The Federal Pell Grant Program is a financial ...
1,How do I apply for a Direct Loan?,"To apply for a Direct Loan, you should complet...",[complete the Direct PLUS Loan Application (fo...,"To apply for a Direct Loan, you should complet..."
2,What is the maximum Pell Grant amount?,The maximum Pell Grant amount mentioned in the...,"[programs will be $7,500:\n$15,000\n- 0\n- $7,...",The maximum Pell Grant amount mentioned in the...
3,What are the eligibility requirements for fede...,The eligibility requirements for federal stude...,[Chapter 1\nStudent and Parent Eligibility for...,The eligibility requirements for federal stude...
4,How is the cost of attendance calculated?,The cost of attendance (COA) is calculated bas...,[Chapter 2\nCost of Attendance (Budget)\nAward...,The cost of attendance (COA) is calculated bas...


baseline evaluation

In [87]:
def evaluate_rag_system_fixed(dataset: Dataset, system_name: str) -> Dict[str, float]:
    from ragas.llms import LangchainLLMWrapper
    
    evaluator_llm = LangchainLLMWrapper(llm)

    evaluation_data = []
    for i in range(len(dataset)):
        # ground truth
        question = dataset["question"][i]
        context = dataset["contexts"][i]
        
        # reference answer
        reference_prompt = f"""Based on the following context, provide a comprehensive and accurate answer to the question.

Context: {' '.join(context)}

Question: {question}

Reference Answer:"""
        
        try:
            reference_response = llm.invoke(reference_prompt)
            reference_answer = reference_response.content
        except Exception:
            reference_answer = dataset["answer"][i]
        
        evaluation_data.append({
            "question": question,
            "answer": dataset["answer"][i],
            "contexts": context,
            "reference": reference_answer
        })
    
    evaluation_dataset = Dataset.from_list(evaluation_data)
    
    metrics = [
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
        answer_correctness
    ]
    
    try:
        results = evaluate(
            dataset=evaluation_dataset,
            metrics=metrics,
            llm=evaluator_llm
        )
        
        scores = {}
        for metric in metrics:
            metric_name = metric.name
            if metric_name in results:
                scores[metric_name] = results[metric_name]
        
        print(f"\n=== {system_name} Evaluation Results ===")
        for metric, score in scores.items():
            print(f"{metric}: {score:.4f}")
        
        return scores
        
    except Exception as e:
        print(f"Error evaluating {system_name}: {str(e)}")
        print(f"Error type: {type(e)}")
        import traceback
        traceback.print_exc()
        return {}

compare and analyze

In [88]:
# Alternative simpler evaluation approach
def evaluation_comparison():
    """
    A simpler evaluation approach that compares the systems based on 
    chunk statistics and response quality indicators
    """
    print("=== SIMPLE EVALUATION COMPARISON ===")
    
    # Compare chunk statistics
    print(f"\nChunk Efficiency:")
    print(f"Naive chunks: {len(naive_chunks)} (avg {np.mean([len(c.page_content.split()) for c in naive_chunks]):.1f} words)")
    print(f"Semantic chunks: {len(semantic_chunks)} (avg {np.mean([len(c.page_content.split()) for c in semantic_chunks]):.1f} words)")
    print(f"Reduction: {((len(naive_chunks) - len(semantic_chunks)) / len(naive_chunks) * 100):.1f}%")
    
    # Compare response characteristics
    naive_responses = [synthetic_dataset["naive_answer"][i] for i in range(len(synthetic_dataset))]
    semantic_responses = [synthetic_dataset["semantic_answer"][i] for i in range(len(synthetic_dataset))]
    
    print(f"\nResponse Analysis:")
    print(f"Naive avg response length: {np.mean([len(r.split()) for r in naive_responses]):.1f} words")
    print(f"Semantic avg response length: {np.mean([len(r.split()) for r in semantic_responses]):.1f} words")
    
    # Count "I cannot answer" type responses
    cannot_answer_phrases = ["cannot answer", "does not contain", "does not specify", "not enough information"]
    
    naive_cannot_answer = sum(1 for r in naive_responses if any(phrase in r.lower() for phrase in cannot_answer_phrases))
    semantic_cannot_answer = sum(1 for r in semantic_responses if any(phrase in r.lower() for phrase in cannot_answer_phrases))
    
    print(f"Naive 'cannot answer' responses: {naive_cannot_answer}/{len(naive_responses)}")
    print(f"Semantic 'cannot answer' responses: {semantic_cannot_answer}/{len(semantic_responses)}")
    
    # Context relevance analysis
    print(f"\nContext Analysis:")
    naive_context_lengths = [len(' '.join(c)) for c in synthetic_dataset["naive_context"]]
    semantic_context_lengths = [len(' '.join(c)) for c in synthetic_dataset["semantic_context"]]
    
    print(f"Naive avg context length: {np.mean(naive_context_lengths):.0f} characters")
    print(f"Semantic avg context length: {np.mean(semantic_context_lengths):.0f} characters")
    
    # Overall assessment
    print(f"\nOverall Assessment:")
    if semantic_cannot_answer < naive_cannot_answer:
        print("✅ Semantic RAG provides more complete answers")
    elif semantic_cannot_answer > naive_cannot_answer:
        print("❌ Naive RAG provides more complete answers")
    else:
        print("➖ Both systems provide similar answer completeness")
    
    print(f"✅ Semantic RAG uses {((len(naive_chunks) - len(semantic_chunks)) / len(naive_chunks) * 100):.1f}% fewer chunks")
    print(f"✅ Semantic chunks are {(np.mean([len(c.page_content.split()) for c in semantic_chunks]) / np.mean([len(c.page_content.split()) for c in naive_chunks])):.1f}x larger on average")

# Run the simple evaluation
evaluation_comparison()

=== SIMPLE EVALUATION COMPARISON ===

Chunk Efficiency:
Naive chunks: 1102 (avg 140.3 words)
Semantic chunks: 292 (avg 467.7 words)
Reduction: 73.5%

Response Analysis:
Naive avg response length: 160.9 words
Semantic avg response length: 183.2 words
Naive 'cannot answer' responses: 0/10
Semantic 'cannot answer' responses: 1/10

Context Analysis:
Naive avg context length: 4332 characters
Semantic avg context length: 14062 characters

Overall Assessment:
❌ Naive RAG provides more complete answers
✅ Semantic RAG uses 73.5% fewer chunks
✅ Semantic chunks are 3.3x larger on average


detailed analysis

In [68]:
def analyze_chunking_strategies():
    
    print("=== CHUNKING STRATEGY ANALYSIS ===")
    
    naive_chunk_sizes = [len(chunk.page_content.split()) for chunk in naive_chunks]
    semantic_chunk_sizes = [len(chunk.page_content.split()) for chunk in semantic_chunks]
    
    print(f"\nChunk Size Statistics:")
    print(f"Naive Chunks:")
    print(f"  - Count: {len(naive_chunks)}")
    print(f"  - Average size: {np.mean(naive_chunk_sizes):.1f} words")
    print(f"  - Min size: {min(naive_chunk_sizes)} words")
    print(f"  - Max size: {max(naive_chunk_sizes)} words")
    
    print(f"\nSemantic Chunks:")
    print(f"  - Count: {len(semantic_chunks)}")
    print(f"  - Average size: {np.mean(semantic_chunk_sizes):.1f} words")
    print(f"  - Min size: {min(semantic_chunk_sizes)} words")
    print(f"  - Max size: {max(semantic_chunk_sizes)} words")
    
    print(f"\nContent Analysis:")
    print(f"Naive chunks tend to split at arbitrary character boundaries")
    print(f"Semantic chunks group semantically related content together")
    
    print(f"\nExample Naive Chunk:")
    print(f"Length: {len(naive_chunks[0].page_content.split())} words")
    print(f"Content: {naive_chunks[0].page_content[:500]}...")
    
    print(f"\nExample Semantic Chunk:")
    print(f"Length: {len(semantic_chunks[0].page_content.split())} words")
    print(f"Content: {semantic_chunks[0].page_content[:500]}...")

analyze_chunking_strategies()

=== CHUNKING STRATEGY ANALYSIS ===

Chunk Size Statistics:
Naive Chunks:
  - Count: 1102
  - Average size: 140.3 words
  - Min size: 25 words
  - Max size: 190 words

Semantic Chunks:
  - Count: 292
  - Average size: 467.7 words
  - Min size: 1 words
  - Max size: 922 words

Content Analysis:
Naive chunks tend to split at arbitrary character boundaries
Semantic chunks group semantically related content together

Example Naive Chunk:
Length: 143 words
Content: Volume 3
Academic Calendars, Cost of Attendance, and
Packaging
Introduction
This volume of the Federal Student Aid (FSA) Handbook discusses the academic calendar, payment period, and
disbursement requirements for awarding aid under the Title IV student financial aid programs, determining a student9s
cost of attendance, and packaging Title IV aid.
Throughout this volume of the Handbook, the words "we," "our," and "us" refer to the United States Department of
Education (the Department). The word "...

Example Semantic Chunk:
Length:

test it out!

In [60]:
def test_rag_systems():
    """Interactive testing of both RAG systems"""
    
    test_questions = [
        "What is the maximum Pell Grant amount for 2024?",
        "How do I apply for federal student aid?",
        "What are the verification requirements?",
        "How is financial need calculated?"
    ]
    
    print("=== INTERACTIVE RAG SYSTEM TESTING ===")
    
    for i, question in enumerate(test_questions, 1):
        print(f"\n--- Test Question {i} ---")
        print(f"Question: {question}")
        
        # Get responses from both systems
        naive_response = naive_rag_chain.invoke(question)
        semantic_response = semantic_rag_chain.invoke(question)
        
        print(f"\nNaive RAG Answer:")
        print(f"{naive_response.content}")
        
        print(f"\nSemantic RAG Answer:")
        print(f"{semantic_response.content}")
        
        print("-" * 50)

# Run interactive testing
test_rag_systems()

=== INTERACTIVE RAG SYSTEM TESTING ===

--- Test Question 1 ---
Question: What is the maximum Pell Grant amount for 2024?

Naive RAG Answer:
The context provided does not contain information about the maximum Pell Grant amount for 2024. It mentions maximum Pell Grant amounts for illustrative purposes and discusses the 2025-26 award year, but does not specify the maximum amount for 2024.

Semantic RAG Answer:
The context provided does not specify the maximum Pell Grant amount for 2024. It mentions that the maximum Pell Grant award amount is determined by the appropriation Act applicable to that award year, but it does not provide the specific amount for 2024. Therefore, I cannot answer the question based on the provided context.
--------------------------------------------------

--- Test Question 2 ---
Question: How do I apply for federal student aid?

Naive RAG Answer:
To apply for federal student aid, you need to complete the Free Application for Federal Student Aid (FAFSA®) form. Th